# 01 — Preprocessing

Order aggregation, oversized-stop detection, and the canonical
virtual-stop mapping (the split-stop bugfix's single source of truth).
All logic lives in `src/preprocessing.py` and `src/data.py`; this
notebook only orchestrates and audits.

**PUBLIC mode** demonstrates this code path on `data_demo_synthetic/` --
fully synthetic, fake data -- since real customer-level operational data
is not distributed in this repository (see the data-minimization
rationale in `REPRODUCIBILITY_NOTE.md`). It therefore does NOT assert
the exact manuscript counts (3,927 source stops, 6 oversized, 13 virtual
pieces, 3,934 total); those exact counts are documented as a fixed,
disclosed fact in `configs/experiment_config.json`'s
`oversized_stop_split` field, independently re-derivable only under
**PRIVATE/Level-2** reproduction with the real authorized input.

**PRIVATE mode** runs the same code on the real (authorized, non-public)
raw stops table and DOES assert the exact manuscript counts.


In [ ]:
import os, sys, json
import pandas as pd
assert 'REPO_ROOT' in dir(), "Run notebook 00 first (or re-execute its setup cells)."
sys.path.insert(0, REPO_ROOT)
from src.data import build_canonical_mapping, verify_archived_ids_resolve


## Scope note (explicit, not silently assumed)

In [ ]:
if DATA_MODE == "public":
    print("DATA_MODE=public: real customer-level operational data is NOT")
    print("distributed in this repository (data minimization -- see")
    print("REPRODUCIBILITY_NOTE.md). This notebook instead runs the exact same")
    print("preprocessing code on data_demo_synthetic/, which is fully synthetic")
    print("(randomly generated, not derived from any real source). The published,")
    print("documented manuscript counts (3,927 / 6 / 13 / 3,934) are recorded in")
    print("configs/experiment_config.json as a disclosed fact, not independently")
    print("re-derived here.")
    STOPS_PATH = os.path.join(REPO_ROOT, "data_demo_synthetic", "customer_stops",
                               "synthetic_customer_day_stops.csv")
else:
    print("DATA_MODE=private: expecting an authorized raw order table under data_private/.")
    STOPS_PATH = os.path.join(REPO_ROOT, "data_private", "customer_day_stops_preprocessed.csv")
    if not os.path.exists(STOPS_PATH):
        raise FileNotFoundError(
            "Private raw stops file not found. Full computational reproduction "
            "(Level 2) requires authorized operational inputs -- see README.md.")

print(f"STOPS_PATH = {STOPS_PATH}")


## Build the canonical virtual-stop mapping

In [ ]:
mapping_df, customers, oversized = build_canonical_mapping(STOPS_PATH)

n_source_stops = len(pd.read_csv(STOPS_PATH))
n_oversized = len(oversized)
n_virtual = len(mapping_df)
n_split_pieces = len(mapping_df[mapping_df['piece_count'] > 1])

print(f"Source customer-day stops:        {n_source_stops}")
print(f"Oversized source stops detected:  {n_oversized}")
print(f"Virtual pieces from splits:       {n_split_pieces}")
print(f"Total optimization stops:         {n_virtual}")

if DATA_MODE == "public":
    with open(os.path.join(REPO_ROOT, "configs", "experiment_config.json")) as f:
        exp_cfg = json.load(f)
    doc = exp_cfg["oversized_stop_split"]
    print(f"\n(Documented manuscript counts, PRIVATE/Level-2 fact, NOT asserted "
          f"against this synthetic-data run): source_stops={doc.get('n_source_stops')}, "
          f"n_virtual_stops={doc.get('n_virtual_stops')}, "
          f"total_optimization_stops={doc.get('total_optimization_stops')}")


## Integrity assertions (hard-fail, no silent fallback)

In [ ]:
if DATA_MODE == "private":
    assert n_source_stops == 3927, f"Expected 3,927 source stops, got {n_source_stops}"
    assert n_oversized == 6, f"Expected 6 oversized source stops, got {n_oversized}"
    assert n_split_pieces == 13, f"Expected 13 virtual split stops, got {n_split_pieces}"
    assert n_virtual == 3934, f"Expected 3,934 total optimization stops, got {n_virtual}"
else:
    # Public/synthetic mode: assert STRUCTURAL correctness (the split logic
    # ran and produced a consistent, non-trivial result) rather than the
    # exact manuscript counts, which depend on the real private input.
    assert n_oversized >= 1, "Synthetic demo data should include at least one oversized stop"
    assert n_split_pieces >= 2, "Synthetic demo data's oversized stop should split into >=2 pieces"
    assert n_virtual == n_source_stops - n_oversized + n_split_pieces

# Every mapping row must have a resolvable parent_physical_node (no unresolved IDs).
unresolved_ids = mapping_df[mapping_df['parent_physical_node'].isna()]
print(f"Unresolved IDs: {len(unresolved_ids)} (expected 0)")
assert len(unresolved_ids) == 0

# Matrix resolution check: every parent_physical_node must appear in the
# distance matrix (origin or destination side) for its date.
if DATA_MODE == "public":
    dist_path = os.path.join(REPO_ROOT, "data_demo_synthetic", "matrices", "synthetic_distance_matrix.csv")
else:
    dist_path = os.path.join(REPO_ROOT, "data_private", "combined_osrm_distance_matrix_km_long.csv")
dist_long = pd.read_csv(dist_path)
valid_origin = dist_long.groupby('delivery_date')['origin_id'].apply(set).to_dict()
valid_dest = dist_long.groupby('delivery_date')['destination_id'].apply(set).to_dict()

matrix_failures = []
for _, row in mapping_df.iterrows():
    d, p = row['delivery_date'], row['parent_physical_node']
    if p == "DEPOT_1":
        continue
    if not ((d in valid_origin and p in valid_origin[d]) or (d in valid_dest and p in valid_dest[d])):
        matrix_failures.append((d, row['virtual_stop_id'], p))

print(f"Matrix resolution failures: {len(matrix_failures)} (expected 0)")
assert len(matrix_failures) == 0, f"Sample failures: {matrix_failures[:5]}"

print("\nAll preprocessing integrity assertions: PASS")


## Save the canonical mapping (machine-readable output)

In [ ]:
results_dir = os.path.join(REPO_ROOT, "results")
os.makedirs(results_dir, exist_ok=True)
out_path = os.path.join(results_dir, "canonical_virtual_stop_mapping.csv")
mapping_df.to_csv(out_path, index=False)
print(f"Saved {out_path} ({len(mapping_df)} rows)")


## Expected outputs / integrity checks (summary)

In [ ]:
checks = {
    "unresolved_ids_0": len(unresolved_ids) == 0,
    "matrix_failures_0": len(matrix_failures) == 0,
    "split_logic_produced_pieces": n_split_pieces >= 1,
}
if DATA_MODE == "private":
    checks.update({
        "source_stops_3927": n_source_stops == 3927,
        "oversized_stops_6": n_oversized == 6,
        "virtual_pieces_13": n_split_pieces == 13,
        "total_stops_3934": n_virtual == 3934,
    })
for k, v in checks.items():
    print(f"{'PASS' if v else 'FAIL'}  {k}")
NOTEBOOK_01_STATUS = "PASS" if all(checks.values()) else "FAIL"
print(f"\nNOTEBOOK 01 STATUS: {NOTEBOOK_01_STATUS}")
assert NOTEBOOK_01_STATUS == "PASS"
